In [ ]:
# ============================================================
# 02. PREPROCESSING
# ============================================================

import pandas as pd
import numpy as np

df = pd.read_csv("C:\\Users\\alexa\\Desktop\\customer-churn-prediction\\data\\raw\\telco_customer_churn.csv")

# Cleaning
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna(subset=["TotalCharges"]).copy()

# Drop irrelevant columns BEFORE encoding
df = df.drop(columns=["customerID"])

#No data leakage was identified among the predictor variables. Since all features are available prior to the churn event, 
# categorical variables can be encoded before splitting the dataset into training and test sets.

In [17]:
# ============================================================
# 2. ENCODE TARGET VARIABLE
# ============================================================
#
# Churn is currently stored as "Yes"/"No" strings.
# Machine learning models require numerical values.
#
# Yes -> 1 (churned)
# No  -> 0 (retained)
# ============================================================

df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

print(f"Churn value counts:")
print(df["Churn"].value_counts())
print(f"\nChurn dtype: {df['Churn'].dtype}")

Churn value counts:
Churn
0    5163
1    1869
Name: count, dtype: int64

Churn dtype: int64


In [18]:
# ============================================================
# 3. ENCODE CATEGORICAL VARIABLES
# ============================================================
#
# Machine learning models require numerical input.
# We use pd.get_dummies() for one-hot encoding.
#
# drop_first=True removes one category per variable to avoid
# the dummy variable trap (perfect multicollinearity).
# For example, gender becomes one column: gender_Male
# where 0 = Female and 1 = Male.
# ============================================================

df_encoded = pd.get_dummies(df, drop_first=True)

print(f"Columns before encoding: {df.shape[1]}")
print(f"Columns after encoding:  {df_encoded.shape[1]}")
print(f"\nNew columns:")
print(df_encoded.columns.tolist())

Columns before encoding: 20
Columns after encoding:  31

New columns:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn', 'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year', 'PaperlessBilling_Yes', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


In [19]:
# ============================================================
# 4. TRAIN-TEST SPLIT
# ============================================================
#
# The dataset is divided into:
# - Training set (80%)
# - Test set (20%)
#
# The training set is used for model development.
# The test set remains unseen until final evaluation.
#
# Stratification preserves the original churn distribution
# across both subsets.
# ============================================================

from sklearn.model_selection import train_test_split

# Features
X = df_encoded.drop("Churn", axis=1)

# Target
y = df_encoded["Churn"]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (5625, 30)
X_test : (1407, 30)
y_train: (5625,)
y_test : (1407,)


In [20]:
# ============================================================
# 5. TARGET DISTRIBUTION AFTER SPLIT
# ============================================================

print("Train distribution:")
print(y_train.value_counts(normalize=True).round(4) * 100)

print("\nTest distribution:")
print(y_test.value_counts(normalize=True).round(4) * 100)

Train distribution:
Churn
0    73.42
1    26.58
Name: proportion, dtype: float64

Test distribution:
Churn
0    73.42
1    26.58
Name: proportion, dtype: float64


The train-test split successfully preserved the original class distribution through stratified sampling.

Both subsets maintain the same churn proportion (73.42% retained customers and 26.58% churned customers), ensuring that model training and evaluation are performed on representative samples of the population.

In [21]:
# ============================================================
# 6. SAVE PROCESSED DATASETS
# ============================================================

X_train.to_csv(
    "../data/processed/X_train.csv",
    index=False
)

X_test.to_csv(
    "../data/processed/X_test.csv",
    index=False
)

y_train.to_csv(
    "../data/processed/y_train.csv",
    index=False
)

y_test.to_csv(
    "../data/processed/y_test.csv",
    index=False
)

print("Processed datasets saved.")

Processed datasets saved.
